In [1]:
from spicexplorer.optimization      import Circuit_Optimizer_Orchestrator_with_SPICE
from spicexplorer.viz               import Optimization_Log_Visualizer
from spicexplorer.core.domains      import OptimizationLog, OptimizationLogEntry
from spicexplorer.logging.logger_setup               import setup_loggers_with_spicelib_suppression as setup_loggers

from pathlib import Path
from typing import List, Dict, Tuple, Any

import os
import re

logger = setup_loggers()
logger.info("Plotting Notebool imports completed.")

15:20:27 - spicexplorer: [INFO] 🚀 Logger initialized!
15:20:27 - spicexplorer: [INFO] 📄 Log file: /home/noorizad/code/SpiceXplorer/examples/OTA/cascode/ihp-sg13g2/sizing/logs/SpiceXplorer_2026-02-08_15-20-27.log
15:20:27 - spicexplorer: [INFO] 👀 TQDM and Errors will still show in console (stderr).
15:20:27 - spicexplorer: [INFO] 🔇 Standard outputs and SPICE logs are redirected to file (stdout).
15:20:27 - spicexplorer: [INFO] Plotting Notebool imports completed.


# Helper Functions

In [7]:
def find_project_folders(path: str | Path) -> List[Path]:
    # Matches: project_optimizer_YYYY-MM-DD_HH-MM-SS
    pattern = re.compile(r'^[^_]+_[^_]+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$')
    
    matches: List[Path] = []
    if not os.path.exists(path):
        return matches

    for entry in os.scandir(path):
        if entry.is_dir() and pattern.match(entry.name):
            matches.append(Path(entry.path))
            
    return matches

def find_project_folders_nested(base_path: str | Path) -> Dict[str, Dict[str, Dict[str, Path]]]:
    """
    Returns a nested dictionary: {project_name: {optimizer: [(timestamp: path)]}}
    """
    # Pattern: project_optimizer_YYYY-MM-DD_HH-MM-SS
    pattern = re.compile(r'^([^_]+)_([^_]+)_(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})$')
    
    path_obj = Path(base_path)
    nested_results: Dict[str, Dict[str, Dict[str, Path]]] = {}

    if not path_obj.exists():
        return nested_results

    # Use iterdir() for Path-based iteration
    for folder in path_obj.iterdir():
        if folder.is_dir():
            match = pattern.match(folder.name)
            if match:
                # Capture groups from the regex
                project, optimizer, timestamp = match.groups()
                
                # Initialize nested levels if they don't exist
                if project not in nested_results:
                    nested_results[project] = {}
                if optimizer not in nested_results[project]:
                    nested_results[project][optimizer] = {}
                
                # Assign the Path object
                nested_results[project][optimizer][timestamp] = folder
                
    return nested_results

def extract_json_data(target_dirs: List[Path]) -> Dict[str, List[Path]]:
    combined_data: Dict[str, List[Path]] = {}
    for dir_path in target_dirs:
        if not isinstance(dir_path, Path): dir_path = Path(dir_path)
        combined_data[dir_path.name] = [Path(file.path )for file in os.scandir(dir_path) if (file.is_file() and file.name.endswith('.json'))]
        logger.info(f"directory {dir_path.name} has {len(combined_data[dir_path.name])} json files")
            
    return combined_data



In [8]:
def get_runs(project: str, optimizer: str, base_path: str | Path) -> List[Path]:
    nested_results = find_project_folders_nested(base_path=base_path)
    return list(nested_results[project][optimizer].values())

In [9]:
def load_from_json_list(json_path_list) -> Optimization_Log_Visualizer:
    viz = Optimization_Log_Visualizer(optimization_log=OptimizationLog(initial_logs=[]))
    logger.info(f"initial viz size: {len(viz.optimization_log.log)}")
    for path_to_checkpoint in json_path_list:
        loaded_ckpt = Optimization_Log_Visualizer.load_checkpoint(path_to_checkpoint=path_to_checkpoint)
        logger.info(f"initial loaded size: {len(loaded_ckpt.optimization_log.log)}")
        for entry in loaded_ckpt.optimization_log.log:
            viz.optimization_log.append(entry=entry)

        logger.info(f"current loaded size: {len(viz.optimization_log.log)}")
    logger.info(f"final viz size: {len(viz.optimization_log.log)}")

    return viz

# Loading Results

## Option 1

In [ ]:
# Option (1) to load
# -------------------------------- 

# target_dirs: List[Path] = find_project_folders('./auto_save')
# print(target_dirs)
# results = extract_json_data(target_dirs=target_dirs)

[PosixPath('auto_save/CASCODE-OTA_TinyCMA_2026-02-07_10-51-36'), PosixPath('auto_save/CASCODE-OTA_LHSSearch_2026-02-07_10-53-21'), PosixPath('auto_save/CASCODE-OTA_LhsDE_2026-02-07_10-54-54'), PosixPath('auto_save/CASCODE-OTA_LogMultiBFGSPlus_2026-02-07_10-58-00'), PosixPath('auto_save/CASCODE-OTA_SQOPSO_2026-02-07_11-18-50'), PosixPath('auto_save/CASCODE-OTA_NGOpt_2026-02-07_11-25-10'), PosixPath('auto_save/CASCODE-OTA_LhsDE_2026-02-07_14-53-23'), PosixPath('auto_save/CASCODE-OTA_LhsDE_2026-02-07_15-10-21'), PosixPath('auto_save/CASCODE-OTA_LhsDE_2026-02-07_18-23-01'), PosixPath('auto_save/CASCODE-OTA_LBFGSB_2026-02-07_23-59-42'), PosixPath('auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-38'), PosixPath('auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55')]
directory CASCODE-OTA_TinyCMA_2026-02-07_10-51-36 has 1 json files
directory CASCODE-OTA_LHSSearch_2026-02-07_10-53-21 has 5 json files
directory CASCODE-OTA_LhsDE_2026-02-07_10-54-54 has 1 json files
directory CASCODE

## Option 2

In [38]:
# Option (2) to load
# -------------------------------- 

# targets_for_optimizer = get_runs(project="CASCODE-OTA", optimizer="LHSSearch", base_path='./auto_save')
# targets_for_optimizer = get_runs(project="CASCODE-OTA", optimizer="LogMultiBFGSPlus", base_path='./auto_save')
targets_for_optimizer = get_runs(project="CASCODE-OTA", optimizer="LogBFGSCMAPlus", base_path='./auto_save')
# targets_for_optimizer = get_runs(project="CASCODE-OTA", optimizer="LhsDE", base_path='./auto_save')
# targets_for_optimizer = get_runs(project="CASCODE-OTA", optimizer="TinyCMA", base_path='./auto_save')

results_for_optimizer = extract_json_data(target_dirs=targets_for_optimizer)
results_for_optimizer

15:35:20 - spicexplorer: [INFO] directory CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-38 has 0 json files
15:35:20 - spicexplorer: [INFO] directory CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55 has 1 json files


{'CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-38': [],
 'CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55': [PosixPath('auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55/CASCODE-OTA_LogBFGSCMAPlus_2000_trial2000_FINAL_2026-02-08_03-07-05.json')]}

In [39]:
# idx = "CASCODE-OTA_LHSSearch_2026-02-07_10-53-21"

# idx = "CASCODE-OTA_LhsDE_2026-02-07_10-54-54"
# idx = "CASCODE-OTA_LhsDE_2026-02-07_14-53-23"

# idx = "CASCODE-OTA_TinyCMA_2026-02-07_10-51-36"

# idx = "CASCODE-OTA_LogMultiBFGSPlus_2026-02-07_10-58-00"
idx = "CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55"


# json_path_list  = results[idx]
json_path_list  = results_for_optimizer[idx]
json_path       = json_path_list[-1]

logger.info(f"selected run = {idx}")
logger.info(f"selected path:  {json_path}")
logger.info(f"list (len {len(json_path_list)}) : {json_path_list}")

15:35:44 - spicexplorer: [INFO] selected run = CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55
15:35:44 - spicexplorer: [INFO] selected path:  auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55/CASCODE-OTA_LogBFGSCMAPlus_2000_trial2000_FINAL_2026-02-08_03-07-05.json
15:35:44 - spicexplorer: [INFO] list (len 1) : [PosixPath('auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55/CASCODE-OTA_LogBFGSCMAPlus_2000_trial2000_FINAL_2026-02-08_03-07-05.json')]


## Constructing the Optimization_Log_Visualizer object

In [40]:
# viz = Optimization_Log_Visualizer.load_checkpoint(path_to_checkpoint=json_path)
# logger.info(f"loaded count {len(viz.optimization_log)}")

viz = load_from_json_list(json_path_list=json_path_list)

15:35:46 - spicexplorer: [INFO] initial viz size: 0
15:35:47 - spicexplorer.viz.plotting: [INFO] ✅ Checkpoint loaded successfully from auto_save/CASCODE-OTA_LogBFGSCMAPlus_2026-02-08_00-01-55/CASCODE-OTA_LogBFGSCMAPlus_2000_trial2000_FINAL_2026-02-08_03-07-05.json
15:35:47 - spicexplorer: [INFO] initial loaded size: 2000
15:35:47 - spicexplorer: [INFO] current loaded size: 2000
15:35:47 - spicexplorer: [INFO] final viz size: 2000


## Exporting to CSV and Pandas Dataframe

In [41]:
description = "BFGS_relativeAbs-loss"
csv_filename = f"{idx}_{description}.csv"
viz_df = viz.to_df()
viz_df.to_csv(csv_filename)

viz_df

,point.params.X_DUT_M1M2_W,point.params.X_DUT_M1M2_L,point.params.X_DUT_M1CM2C_W,point.params.X_DUT_M1CM2C_L,point.params.X_DUT_M3M4_W,point.params.X_DUT_M3M4_L,point.params.X_DUT_M3CM4C_W,point.params.X_DUT_M3CM4C_L,point.params.X_DUT_M5_W,point.params.X_DUT_M5_L,...,fit_summary.dcgain.curr_val,fit_summary.dcgain.score,fit_summary.pm.curr_val,fit_summary.pm.score,fit_summary.v(inoise_total).curr_val,fit_summary.v(inoise_total).score,fit_summary.i(idd_total).curr_val,fit_summary.i(idd_total).score,fit_summary.tsettle.curr_val,fit_summary.tsettle.score
0,0.000093,5.579081e-06,0.000101,5.332491e-06,0.000108,5.835088e-06,0.000091,5.803696e-06,0.000117,5.846409e-06,...,31.669040,-7.330960,96.0,-57.777778,0.000808,17.164223,0.000010,40.553157,3.189807e-06,0.0
1,0.000090,4.615382e-06,0.000095,6.412098e-06,0.000094,4.997999e-06,0.000103,5.072613e-06,0.000091,5.366615e-06,...,29.174110,-9.825890,89.0,-42.222222,0.000853,14.798650,0.000008,51.648919,5.295431e-06,0.0
2,0.000090,4.718465e-06,0.000106,5.421440e-06,0.000085,5.446409e-06,0.000095,5.028087e-06,0.000098,4.906957e-06,...,30.595930,-8.404070,88.0,-40.000000,0.000796,17.848408,0.000009,46.274959,3.986854e-06,0.0
3,0.000018,8.031328e-06,0.000141,9.339014e-06,0.000089,5.134941e-07,0.000057,4.511613e-06,0.000065,5.553321e-06,...,-18.494980,-57.494980,NaN,-1000000.000000,0.000242,69.486224,0.000005,69.713372,NaN,-1000000.0
4,0.000153,2.536505e-06,0.000195,2.609188e-07,0.000076,9.604532e-06,0.000072,5.086278e-06,0.000036,6.731665e-06,...,7.438343,-31.561657,62.0,0.000000,0.003136,-193.549905,0.000006,64.214401,4.429769e-06,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.000186,8.599067e-06,0.000020,9.948471e-06,0.000168,8.846809e-06,0.000122,5.681000e-06,0.000045,3.364091e-06,...,-3.217693,-42.217693,NaN,-1000000.000000,0.000409,46.747407,0.000006,63.053708,NaN,-1000000.0
1996,0.000018,5.325734e-06,0.000120,9.963193e-06,0.000156,7.336040e-06,0.000140,6.707955e-07,0.000114,6.166672e-07,...,30.738210,-8.261790,91.0,-46.666667,0.000545,34.317671,0.000107,-80.542278,NaN,-1000000.0
1997,0.000058,5.975857e-07,0.000070,9.977916e-06,0.000192,2.048348e-06,0.000031,2.073653e-06,0.000031,1.133509e-06,...,21.722830,-17.277170,86.0,-35.555556,0.003707,-250.618744,0.000007,55.737843,NaN,-1000000.0
1998,0.000069,5.846712e-07,0.000050,5.923435e-07,0.000027,4.597223e-07,0.000104,2.366194e-06,0.000110,8.420218e-06,...,25.861740,-13.138260,58.0,0.000000,0.000644,27.026251,0.000021,7.480793,3.570910e-07,0.0


## Plotting Scores

In [42]:
_ = viz.plot_loss_breakdown(show=True, log_y=False)

In [43]:
_ = viz.plot_best_score_evolution(show=True)

In [44]:
_ = viz.plot_best_score_evolution(show=True)

# Visualize

In [ ]:
viz.list_available_params()

In [ ]:
viz.list_available_metrics()

In [ ]:
viz.plot_design_space_exploration(param_x="X_DUT_M1CM2C_W", param_y="X_DUT_M1CM2C_L", show=True)

In [ ]:
viz.plot_design_space_exploration(param_x="X_DUT_V_BIAS_1", param_y="X_DUT_V_BIAS_2", show=True)

In [ ]:
viz.plot_optimization_trace(metric_x="ugf", metric_y="dcgain", show=True)
viz.plot_optimization_trace(metric_x="ugf", metric_y="dcgain", show=True, log_x=True, log_y=False)
viz.plot_optimization_trace(metric_x="ugf", metric_y="dcgain", show=True, log_x=False, log_y=True)
viz.plot_optimization_trace(metric_x="ugf", metric_y="dcgain", show=True, log_x=True, log_y=True)


In [ ]:
viz.plot_optimization_trace(metric_x="ugf", metric_y="pm", show=True, log_x=True, log_y=False)


In [ ]:
viz.plot_optimization_trace(metric_x="v(inoise_total)", metric_y="dcgain", show=True, log_x=True, log_y=False)


In [ ]:
viz.plot_optimization_trace(metric_x="dcgain", metric_y="i(idd_total)", show=True, log_x=False, log_y=True)
viz.plot_optimization_trace(metric_x="ugf", metric_y="i(idd_total)", show=True, log_x=True, log_y=True)
viz.plot_optimization_trace(metric_x="tsettle", metric_y="dcgain", show=True, log_x=True, log_y=False)
